# 03 — Error Analysis

Goes beyond the aggregate Recall@K numbers: which target-domain queries
failed under zero-shot transfer, which of those recovered after Method B,
and what that says about *why* the domain gap exists (see README §2).

In [ ]:
import sys
sys.path.append("..")

import torch
from src.visualization import (
    plot_domain_embedding_gap, plot_retrieval_comparison,
    build_error_dataframe, plot_error_taxonomy, topk_indices, TAXONOMY
)

zero_shot = torch.load("../results/tables/target_zero_shot_eval_embeddings.pt")
adapted = torch.load("../results/tables/method_b_eval_embeddings.pt")
source = torch.load("../results/tables/source_eval_embeddings.pt")

## Embedding-space view of the domain gap

Source vs. target image embeddings, before and after Method B's MMD alignment.

In [ ]:
plot_domain_embedding_gap(
    source_embeds=source["image_embeds"].numpy(),
    target_embeds_before=zero_shot["image_embeds"].numpy(),
    target_embeds_after=adapted["image_embeds"].numpy(),
    out_path="../results/figures/embedding_domain_gap.png",
)

## Failure taxonomy

For each target-domain query that failed under zero-shot transfer (correct
match not in top-5), bucket the failure and check whether Method B
recovered it. In a full run this loop is driven off the saved embeddings
above plus manual/heuristic labeling; the structure below is what
`build_error_dataframe` expects — replace `example_records` with the real
per-query labels once you've run the pipeline.

In [ ]:
# Placeholder structure — replace with real per-query failure labels
# derived from comparing top-5 ranks in `zero_shot` vs `adapted`.
example_records = [
    {"bucket": "style_shift", "recovered_after_adaptation": True},
    {"bucket": "style_shift", "recovered_after_adaptation": False},
    {"bucket": "granularity_mismatch", "recovered_after_adaptation": False},
    {"bucket": "hard_negative_confusion", "recovered_after_adaptation": True},
    {"bucket": "other", "recovered_after_adaptation": False},
]

summary = build_error_dataframe(example_records)
summary

In [ ]:
plot_error_taxonomy(summary, out_path="../results/figures/error_taxonomy.png")

## Qualitative retrieval grids

Pick a handful of representative queries and show top-5 retrieved images,
baseline vs. adapted, side by side — this is the same rendering logic the
Gradio demo (`app/demo.py`) uses live.

In [ ]:
query_text = "a sketch of a dog"

sim_baseline = zero_shot["text_embeds"] @ zero_shot["image_embeds"].t()
sim_adapted = adapted["text_embeds"] @ adapted["image_embeds"].t()

# NOTE: indices here are illustrative; align to the actual query's row
# in the saved embedding tensors before running for real.
baseline_top = [zero_shot["paths"][i] for i in topk_indices(sim_baseline[0], k=5)]
adapted_top = [adapted["paths"][i] for i in topk_indices(sim_adapted[0], k=5)]

plot_retrieval_comparison(
    query_text=query_text,
    baseline_paths=baseline_top,
    adapted_paths=adapted_top,
    image_root="../data/target",
    out_path="../results/figures/retrieval_comparison_example.png",
)

## Takeaways to write up

- Which taxonomy bucket dominates the failure count, and does Method B's
  MMD term actually target that bucket (it should help most with
  `style_shift`, less with `granularity_mismatch`)?
- Does the embedding-space plot show source/target clusters visibly
  merging post-adaptation, or just shifting without properly overlapping?
- Cross-reference with `experiments/ablations/` results: if
  `ablation_no_mmd` recovers `style_shift` failures almost as well as full
  Method B, that's evidence the adapter's capacity — not the alignment
  term — is doing the work, and the README narrative should say so.